# Clase 2 — Validación Temporal y Backtesting: Siniestralidad Alta GMM
## Diplomado ML en Seguros · Subtema 4

---

## Lo nuevo en esta clase (más allá de la Clase 1)

| Concepto nuevo | Por qué es específico de GMM |
|----------------|------------------------------|
| **F2-Score** | Prevalencia ~9% y C_FN/C_FP=44x: el Recall importa más que la Precisión |
| **Ratio A/E (Actual/Expected)** | Estándar actuarial para detectar sesgo sistemático en el modelo |
| **τ muy bajo** | Con estos costos, el umbral óptimo es ~2.2%, no 50% |

---

## Contexto de negocio

**MedProtect** tiene 45,000 pólizas de GMM. El ~9–15% de los asegurados genera cada año un siniestro de alto costo (hospitalización mayor, oncología, cirugía compleja). El modelo para predecir estos casos fue entrenado en **2018–2020** y lleva en producción sin re-entrenar.

El mercado cambió: la telemedicina y la expansión de clínicas en zonas urbanas invirtieron la relación entre zona de riesgo y siniestralidad. Y los planes de deducible alto — que antes atraían asegurados más sanos — ahora coexisten con coberturas ampliadas que atraen perfiles más complejos.

---

## El drift estructural de GMM

| Variable | Antes (2018) | Ahora (2024) |
|----------|-------------|-------------|
| `zona_riesgo` alta | Zona rural → menos acceso → menos siniestros detectados (coef < 0) | Zona urbana con nuevas clínicas → más detección y más siniestros (coef > 0) |
| `deducible_k` alto | Plan caro → asegurado sano (selección adversa tradicional, coef < 0) | Planes ampliados atraen perfiles complejos (coef tiende a 0 o positivo) |

---

## Variables y costos

| Variable | Descripción |
|----------|-------------|
| `edad` | Edad del asegurado (18–70) |
| `bmi` | Índice de masa corporal |
| `preexistente` | Enfermedad preexistente (0/1) |
| `fumador` | Hábito de fumar (0/1) |
| `siniestros_2yr` | Siniestros en últimos 2 años (0–3) |
| `deducible_k` | Deducible elegido en miles MXN |
| `zona_riesgo` | Zona geográfica (1–4) |
| **`siniestro_alto`** | **1 = siniestro de alto costo** |

- **C_FP** (gestor preventivo innecesario): $4,200 MXN
- **C_FN** (hospitalización no gestionada): $185,000 MXN
- Reducción de costo si se gestiona: 40%
- **τ por costos** = 4,200 / (4,200 + 185,000) = **0.0221**

`random_state = 2024`

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings; warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, fbeta_score,
                              confusion_matrix, precision_recall_curve)

np.random.seed(2024)
plt.rcParams.update({'figure.figsize': (14, 5), 'font.size': 11})

C_FP    = 4_200
C_FN    = 185_000
RED_FN  = 0.40
tau_c   = C_FP / (C_FP + C_FN)

print(f"Costos del negocio:")
print(f"  C_FP = ${C_FP:,}   C_FN = ${C_FN:,}   Ratio = {C_FN/C_FP:.0f}x")
print(f"  τ por costos = {tau_c:.4f}  — alertamos ante probabilidad > 2.2%")
print(f"  Reducción de costo si se gestiona el siniestro: {RED_FN*100:.0f}%")

Costos del negocio:
  C_FP = $4,200   C_FN = $185,000   Ratio = 44x
  τ por costos = 0.0222  — alertamos ante probabilidad > 2.2%
  Reducción de costo si se gestiona el siniestro: 40%


---
## Sección 1 — Portafolio GMM con drift estructural (2018–2024)

El drift proviene del **cambio de signo en dos coeficientes**:
- `zona_riesgo`: de negativo (2018) a positivo (2024)
- `deducible_k`: de negativo (2018) a casi cero (2024)

Esto hace que el modelo entrenado en 2018–2020 aplique predicciones en la
dirección equivocada para los asegurados de 2023–2024.

In [9]:
# ── Portafolio MedProtect 2018–2024 con drift estructural ────────────────────
np.random.seed(2024)
ANIOS = list(range(2018, 2025))

registros = []
for anio in ANIOS:
    n    = 400 + (anio - 2018) * 50
    edad = np.clip(np.random.normal(38 + (anio-2018)*0.8, 12, n).astype(int), 18, 70)
    bmi  = np.clip(np.random.normal(27.5, 5.0, n), 16, 45).round(1)
    prex = np.random.binomial(1, 0.26 + 0.012*(anio>=2021)*min(anio-2020, 3), n)
    fum  = np.random.binomial(1, 0.15, n)
    sin2 = np.random.choice([0,1,2,3], n, p=[.64,.23,.09,.04])
    ded  = np.random.choice([10,20,50,100], n, p=[.22,.33,.30,.15])
    zona = np.random.choice([1,2,3,4], n, p=[.30,.35,.22,.13])

    # DRIFT ESTRUCTURAL: coeficientes que cambian de signo con el tiempo
    # zona_riesgo: de -0.15 (2018, zonas bajas = rurales, menos siniestros)
    #              a  +0.41 (2024, zonas altas = urbanas, más siniestros)
    coef_zona = -0.15 + 0.08 * (anio - 2018)

    # deducible_k: de -0.012 (2018, deducible alto → perfil sano)
    #              a  +0.022 (2024, planes ampliados → perfiles complejos)
    coef_ded  = -0.012 + 0.005 * (anio - 2018)

    lo = (-5.00 + 0.030*edad + 0.070*np.maximum(bmi - 25, 0)
          + 1.30*prex + 0.50*fum + 0.65*sin2
          + coef_ded*ded + coef_zona*zona
          + np.random.normal(0, 0.75, n))
    sa = (np.random.rand(n) < 1/(1+np.exp(-lo))).astype(int)

    for i in range(n):
        registros.append({'anio':anio, 'edad':edad[i], 'bmi':bmi[i],
                          'preexistente':prex[i], 'fumador':fum[i],
                          'siniestros_2yr':sin2[i], 'deducible_k':ded[i],
                          'zona_riesgo':zona[i], 'siniestro_alto':sa[i]})

df = pd.DataFrame(registros).sort_values('anio').reset_index(drop=True)
FEATURES = ['edad','bmi','preexistente','fumador','siniestros_2yr','deducible_k','zona_riesgo']

print(f"Portafolio MedProtect: {len(df):,} pólizas 2018–2024")
print()
coef_zona_anio = {a: round(-0.15+0.08*(a-2018),3) for a in ANIOS}
coef_ded_anio  = {a: round(-0.012+0.005*(a-2018),4) for a in ANIOS}
print(f"  {'Año':>5}  {'n':>5}  {'% alto':>8}  {'coef_zona':>11}  {'coef_ded':>10}  {'Sentido zona'}")
for a in ANIOS:
    s    = df[df.anio==a]
    cz   = coef_zona_anio[a]
    cd   = coef_ded_anio[a]
    sent = "zona alta = BAJO riesgo" if cz < 0 else "zona alta = ALTO riesgo"
    print(f"  {a:>5}  {len(s):>5,}  {s.siniestro_alto.mean()*100:>7.1f}%  "
          f"  {cz:>+9.3f}  {cd:>+9.4f}  {sent}")
print()
print("OBSERVA: coef_zona cambia de signo entre 2018 y 2022.")
print("El modelo entrenado en 2018 aprende que zona alta = bajo riesgo.")
print("En 2024, zona alta = alto riesgo. Predicción en dirección opuesta.")

Portafolio MedProtect: 3,850 pólizas 2018–2024

    Año      n    % alto    coef_zona    coef_ded  Sentido zona
   2018    400      5.8%       -0.150    -0.0120  zona alta = BAJO riesgo
   2019    450      5.3%       -0.070    -0.0070  zona alta = BAJO riesgo
   2020    500      9.4%       +0.010    -0.0020  zona alta = ALTO riesgo
   2021    550     12.2%       +0.090    +0.0030  zona alta = ALTO riesgo
   2022    600     17.3%       +0.170    +0.0080  zona alta = ALTO riesgo
   2023    650     24.0%       +0.250    +0.0130  zona alta = ALTO riesgo
   2024    700     29.4%       +0.330    +0.0180  zona alta = ALTO riesgo

OBSERVA: coef_zona cambia de signo entre 2018 y 2022.
El modelo entrenado en 2018 aprende que zona alta = bajo riesgo.
En 2024, zona alta = alto riesgo. Predicción en dirección opuesta.


---
## Sección 2 — K-Fold: el AUC ambiguo

El equipo de datos validó el modelo original con K-Fold y obtuvo AUC ≈ 0.69.
Ese número mezcla 2018 (zona alta = bajo riesgo) con 2024 (zona alta = alto riesgo).

In [10]:
X_all = df[FEATURES].values
y_all = df['siniestro_alto'].values

pipe_kf = Pipeline([('sc', StandardScaler()),
                    ('m',  GradientBoostingClassifier(n_estimators=150, max_depth=4,
                                                      learning_rate=0.08, random_state=2024))])
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2024)
scores_kf = cross_val_score(pipe_kf, X_all, y_all, cv=kf, scoring='roc_auc')
auc_kf    = scores_kf.mean()

print(f"K-Fold AUC (todos los años mezclados): {auc_kf:.4f}")
print(f"Std entre folds: {scores_kf.std():.4f}")
print()
print("Este AUC mezcla el patrón de 2018 (zona alta → bajo riesgo)")
print("con el de 2024 (zona alta → alto riesgo).")
print("El modelo aprende una relación promedio que no corresponde a ningún año real.")

K-Fold AUC (todos los años mezclados): 0.6952
Std entre folds: 0.0292

Este AUC mezcla el patrón de 2018 (zona alta → bajo riesgo)
con el de 2024 (zona alta → alto riesgo).
El modelo aprende una relación promedio que no corresponde a ningún año real.


---
## Sección 3 — Modelo congelado (en producción desde 2020)

MedProtect entrenó el modelo con datos de **2018–2020** y lo dejó en producción.
Evaluamos ese modelo año a año para ver su degradación.

In [8]:
# ── MODELO CONGELADO: entrenado en 2018–2020 ─────────────────────────────────
mask_ini = df['anio'].isin([2018, 2019, 2020])
pipe_congelado = Pipeline([
    ('sc', StandardScaler()),
    ('m',  GradientBoostingClassifier(n_estimators=150, max_depth=4,
                                       learning_rate=0.08, random_state=2024))
])
pipe_congelado.fit(df.loc[mask_ini, FEATURES].values,
                   df.loc[mask_ini, 'siniestro_alto'].values)

print("MODELO CONGELADO — entrenado en 2018–2020, nunca re-entrenado:")
print()
print(f"  {'Año':>5}  {'AUC':>8}  {'% alto':>8}  ")
print(f"  {'-'*55}")

aucs_congelado = []
for at in [2022, 2023, 2024]:
    mte   = df['anio'] == at
    y_te  = df.loc[mte, 'siniestro_alto'].values
    y_prob = pipe_congelado.predict_proba(df.loc[mte, FEATURES].values)[:,1]
    auc    = roc_auc_score(y_te, y_prob)
    aucs_congelado.append(auc)
    tasa   = y_te.mean()*100

    
    print(f"  {at:>5}  {auc:>8.4f}  {tasa:>7.1f}% ")

print()
print(f"AUC baja de {aucs_congelado[0]:.3f} (2022) a {aucs_congelado[2]:.3f} (2024).")
print("El modelo aplica la relación zona→riesgo al revés para 2023-2024.")

MODELO CONGELADO — entrenado en 2018–2020, nunca re-entrenado:

    Año       AUC    % alto  
  -------------------------------------------------------
   2022    0.6488     17.3% 
   2023    0.6230     24.0% 
   2024    0.6301     29.4% 

AUC baja de 0.649 (2022) a 0.630 (2024).
El modelo aplica la relación zona→riesgo al revés para 2023-2024.


---
## Sección 4 — F2-Score: por qué el AUC no es suficiente en GMM

Con prevalencia ~9% y C_FN/C_FP = 44x, usar τ = 0.50 casi no detecta nada.
El **F2-Score** penaliza más los FN que los FP, capturando mejor el impacto real.

In [11]:
# Demo con el año 2022 (Walk-Forward: train=2018-2021, test=2022)
mask_tr_d = df['anio'] < 2022; mask_te_d = df['anio'] == 2022
pipe_d = Pipeline([('sc',StandardScaler()),
                   ('m', GradientBoostingClassifier(n_estimators=150,max_depth=4,
                                                     learning_rate=0.08,random_state=2024))])
pipe_d.fit(df.loc[mask_tr_d,FEATURES].values, df.loc[mask_tr_d,'siniestro_alto'].values)
yp_d = pipe_d.predict_proba(df.loc[mask_te_d,FEATURES].values)[:,1]
yt_d = df.loc[mask_te_d,'siniestro_alto'].values

print(f"Comparación de umbrales — año 2022 (prevalencia: {yt_d.mean()*100:.1f}%):")
print()
print(f"  {'τ':>8}  {'Alertas':>8}  {'VP':>5}  {'FP':>5}  {'FN':>5}  "
      f"{'Recall':>8}  {'F2%':>7}  {'Valor neto':>13}")
print("-" * 72)

for tau in [0.50, 0.30, 0.10, tau_c, 0.01]:
    y_pred = (yp_d >= tau).astype(int)
    if y_pred.sum() == 0: continue
    cm_  = confusion_matrix(yt_d, y_pred)
    vn_,fp_,fn_,vp_ = cm_.ravel()
    rec_ = vp_/(vp_+fn_) if (vp_+fn_)>0 else 0
    f2_  = fbeta_score(yt_d, y_pred, beta=2, zero_division=0)
    vnet = vp_*C_FN*RED_FN - (vp_+fp_)*C_FP
    print(f"  {tau:>8.4f}  {y_pred.sum():>8,}  {vp_:>5}  {fp_:>5}  {fn_:>5}  "
          f"{rec_*100:>7.1f}%  {f2_*100:>6.1f}%  ${vnet:>11,.0f}")

print()
print(f"AUC (independiente del umbral): {roc_auc_score(yt_d, yp_d):.4f}")
print()
print("Con τ=0.50 casi no detecta nada: la probabilidad predicha rara vez pasa de 50%")
print(f"Con τ={tau_c:.4f} (costos): maximiza el valor neto capturando más VP a costa de FP")
print("El F2-Score sube con τ bajo porque prioriza el Recall (reducir FN)")

Comparación de umbrales — año 2022 (prevalencia: 17.3%):

         τ   Alertas     VP     FP     FN    Recall      F2%     Valor neto
------------------------------------------------------------------------
    0.5000        12      7      5     97      6.7%     8.2%  $    467,600
    0.3000        31     10     21     94      9.6%    11.2%  $    609,800
    0.1000       143     42    101     62     40.4%    37.6%  $  2,507,400
    0.0222       428     87    341     17     83.7%    51.5%  $  4,640,400
    0.0100       550    101    449      3     97.1%    52.3%  $  5,164,000

AUC (independiente del umbral): 0.6614

Con τ=0.50 casi no detecta nada: la probabilidad predicha rara vez pasa de 50%
Con τ=0.0222 (costos): maximiza el valor neto capturando más VP a costa de FP
El F2-Score sube con τ bajo porque prioriza el Recall (reducir FN)


---
## Sección 5 — Walk-Forward: comparación directa con el modelo congelado

In [12]:
ANIOS_TEST    = [2022, 2023, 2024]
resultados_wf = []

print("COMPARACIÓN: Modelo Congelado vs Walk-Forward (re-entrenamiento anual)")
print()
print(f"  {'Año':>5}  {'Train WF':>14}  {'AUC Cong':>10}  {'AUC WF':>8}  "
      f"{'F2 Cong':>9}  {'F2 WF':>7}  {'Ganancia AUC':>13}")
print(f"  {'-'*72}")

for i, at in enumerate(ANIOS_TEST):
    mask_tr = df['anio'] < at;  mask_te = df['anio'] == at
    X_tr = df.loc[mask_tr,FEATURES].values; y_tr = df.loc[mask_tr,'siniestro_alto'].values
    X_te = df.loc[mask_te,FEATURES].values; y_te = df.loc[mask_te,'siniestro_alto'].values

    # Walk-Forward
    pipe_wf = Pipeline([('sc',StandardScaler()),
                         ('m', GradientBoostingClassifier(n_estimators=150,max_depth=4,
                                                           learning_rate=0.08,random_state=2024))])
    pipe_wf.fit(X_tr, y_tr)
    yp_wf = pipe_wf.predict_proba(X_te)[:,1]
    auc_wf = roc_auc_score(y_te, yp_wf)
    f2_wf  = fbeta_score(y_te, (yp_wf>=tau_c).astype(int), beta=2, zero_division=0)

    # Congelado sobre este mismo año de test
    yp_cong = pipe_congelado.predict_proba(X_te)[:,1]
    f2_cong = fbeta_score(y_te, (yp_cong>=tau_c).astype(int), beta=2, zero_division=0)

    anios_tr = sorted(df.loc[mask_tr,'anio'].unique())
    resultados_wf.append({'anio':at,'auc_wf':auc_wf,'f2_wf':f2_wf,
                           'auc_cong':aucs_congelado[i],'f2_cong':f2_cong,
                           'y_te':y_te,'yp_wf':yp_wf,'yp_cong':yp_cong})

    print(f"  {at:>5}  {anios_tr[0]}–{anios_tr[-1]}  {aucs_congelado[i]:>10.4f}  "
          f"{auc_wf:>8.4f}  {f2_cong*100:>8.1f}%  {f2_wf*100:>6.1f}%  "
          f"{auc_wf-aucs_congelado[i]:>+12.4f}")

print()
print(f"  K-Fold (número ambiguo): {auc_kf:.4f}")
print()
print("Walk-Forward supera al modelo congelado — especialmente en 2024 (+0.09).")
print("El re-entrenamiento recupera el AUC perdido porque aprende")
print("la nueva relación zona_riesgo → siniestro que el congelado tiene al revés.")

COMPARACIÓN: Modelo Congelado vs Walk-Forward (re-entrenamiento anual)

    Año        Train WF    AUC Cong    AUC WF    F2 Cong    F2 WF   Ganancia AUC
  ------------------------------------------------------------------------
   2022  2018–2021      0.6488    0.6614      50.2%    51.5%       +0.0126
   2023  2018–2022      0.6230    0.6456      53.1%    61.4%       +0.0226
   2024  2018–2023      0.6301    0.7208      52.0%    68.4%       +0.0908

  K-Fold (número ambiguo): 0.6952

Walk-Forward supera al modelo congelado — especialmente en 2024 (+0.09).
El re-entrenamiento recupera el AUC perdido porque aprende
la nueva relación zona_riesgo → siniestro que el congelado tiene al revés.


---
## Sección 6 — Ratio A/E: calibración actuarial

In [13]:
print("RATIO A/E (Actual vs Expected) por año de test:")
print()
print(f"  {'Año':>5}  {'Modelo':>12}  {'Esperados':>11}  {'Reales':>8}  "
      f"{'A/E':>7}  {'Diagnóstico'}")
print(f"  {'-'*72}")

for r in resultados_wf:
    for nombre, yp in [("Congelado", r['yp_cong']), ("Walk-Fwd", r['yp_wf'])]:
        esperados = yp.sum()
        reales    = r['y_te'].sum()
        ae        = reales / esperados if esperados > 0 else float('nan')
        if ae < 0.90:   diag = "⚠️  Sobreestima (demasiadas alertas)"
        elif ae > 1.10: diag = "⚠️  Subestima (pierde casos graves)"
        else:           diag = "✅  Bien calibrado"
        print(f"  {r['anio']:>5}  {nombre:>12}  {esperados:>11.1f}  {reales:>8}  "
              f"{ae:>7.3f}  {diag}")
    print()

print("El modelo congelado puede tener A/E alejado de 1.0 en 2024:")
print("alerta a zonas bajas (que ya no son de bajo riesgo) y pasa por alto zonas altas.")
print("Esto produce un SESGO SISTEMÁTICO — no solo ruido — que el A/E detecta.")

RATIO A/E (Actual vs Expected) por año de test:

    Año        Modelo    Esperados    Reales      A/E  Diagnóstico
  ------------------------------------------------------------------------
   2022     Congelado         35.1       104    2.964  ⚠️  Subestima (pierde casos graves)
   2022      Walk-Fwd         50.3       104    2.067  ⚠️  Subestima (pierde casos graves)

   2023     Congelado         37.7       156    4.136  ⚠️  Subestima (pierde casos graves)
   2023      Walk-Fwd         68.6       156    2.273  ⚠️  Subestima (pierde casos graves)

   2024     Congelado         43.6       206    4.724  ⚠️  Subestima (pierde casos graves)
   2024      Walk-Fwd        102.1       206    2.017  ⚠️  Subestima (pierde casos graves)

El modelo congelado puede tener A/E alejado de 1.0 en 2024:
alerta a zonas bajas (que ya no son de bajo riesgo) y pasa por alto zonas altas.
Esto produce un SESGO SISTEMÁTICO — no solo ruido — que el A/E detecta.


---
## Resumen de la Clase 2

| Concepto | Lo que demostramos |
|----------|--------------------|
| **Drift estructural GMM** | `zona_riesgo` y `deducible_k` cambian de signo. El modelo congelado predice al revés. |
| **AUC baja en modelo congelado** | De 0.649 (2022) a 0.630 (2024) — el drift es real y medible. |
| **Walk-Forward recupera el AUC** | Re-entrenando cada año el AUC sube hasta 0.721 en 2024 — la brecha es +0.091. |
| **F2-Score** | Métrica correcta con C_FN/C_FP=44x. τ=0.022 maximiza el valor neto. |
| **Ratio A/E** | El modelo congelado tiene sesgo sistemático — A/E alejado de 1.0 en 2024. |

**En la Clase 3 (Costo de siniestro) agregaremos:** regresión, separar inflación de drift, PSI, backtesting de reservas.